# 02 · Silver cleaning

Read persisted Bronze tables, label quality problems, and save accepted readings, quarantine, daily observations, fleet and weather.

Run after `01_bronze_ingestion`. This notebook does not rerun Bronze or use its Python variables. The existing max-minus-min distance policy is unchanged.

## Paths, dates and imports

In [ ]:
from pathlib import Path
from datetime import date
import os

from pyspark.sql import SparkSession, functions as F

project_dir = Path.cwd()
if not (project_dir / "weather.py").exists():
    project_dir = project_dir.parent
project_dir = Path(os.environ.get("FLEET_PROJECT_DIR", project_dir)).resolve()
source_dir = project_dir / "data" / "sources"
output_dir = project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
start_date = date(2026, 8, 1)
end_date = date(2026, 8, 30)

# Databricks already supplies spark. Local Python needs a Delta-enabled session.
if "spark" not in globals():
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.master("local[2]").appName("fleet-activity")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.shuffle.partitions", "2"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    spark.sparkContext.setLogLevel("ERROR")
print("Report period:", start_date, "to", end_date)

## Read the saved Bronze tables

In [ ]:
bronze = spark.read.format("delta").load(str(output_dir / "delta" / "bronze_telemetry"))
fleet = spark.read.format("delta").load(str(output_dir / "delta" / "bronze_fleet"))
weather = spark.read.format("delta").load(str(output_dir / "delta" / "bronze_weather"))

## Clean IDs and read timestamps
`try_to_timestamp` returns null when text cannot be parsed. UTC is set in setup; `event_date` is the UTC day.

In [ ]:
parsed = (bronze
    .withColumn("vehicle_id", F.upper(F.trim("vehicle_id")))
    .withColumn("event_time", F.try_to_timestamp("event_timestamp"))
    .withColumn("event_date", F.to_date("event_time")))
parsed.select("vehicle_id", "event_timestamp", "event_time").show(5, truncate=False)

## Label unusable readings
Read the checks from top to bottom: **the first failed check wins**. A null reason means usable so far. NaN and infinity are numeric special values, not distances. Expected: four rejected records.

In [ ]:
invalid_odometer = (
    F.col("odometer_km").isNull()
    | F.isnan("odometer_km")
    | (F.abs(F.col("odometer_km")) == float("inf"))
    | (F.col("odometer_km") < 0)
)
checked = parsed.withColumn("reject_reason",
    F.when(F.col("_corrupt_record").isNotNull(), "malformed_json")
     .when(F.col("vehicle_id").isNull() | (F.col("vehicle_id") == ""), "missing_vehicle")
     .when(F.col("event_time").isNull(), "invalid_timestamp")
     .when(~F.col("event_date").between(start_date, end_date), "outside_report_period")
     .when(invalid_odometer, "invalid_odometer")
     .otherwise(None))
base_rejects = checked.filter(F.col("reject_reason").isNotNull())
valid = checked.filter(F.col("reject_reason").isNull())
base_rejects.groupBy("reject_reason").count().show()

## Remove identical readings
Compare only the business fields. Two copies of the same reading should count once. Expected: one duplicate removed.

In [ ]:
business_columns = ["vehicle_id", "event_time", "event_date", "odometer_km"]
unique_readings = valid.select(*business_columns).dropDuplicates()
exact_duplicates_removed = valid.count() - unique_readings.count()
print("Exact duplicates removed:", exact_duplicates_removed)

## Find conflicting timestamps
After exact duplicates are removed, two readings at the same vehicle/time disagree. `left_anti` keeps rows with no match: leave these conflicts out before ordering readings.

In [ ]:
conflicting_keys = (unique_readings.groupBy("vehicle_id", "event_time")
    .count().filter(F.col("count") > 1).drop("count"))
conflicts = unique_readings.join(conflicting_keys, ["vehicle_id", "event_time"], "inner")
ordered_readings = unique_readings.join(conflicting_keys, ["vehicle_id", "event_time"], "left_anti")
conflicts.orderBy("vehicle_id", "event_time").show(truncate=False)

## Compare each reading with the previous one
Use SQL `LAG`, partitioned by vehicle and UTC day. A falling pair tells us the day is inconsistent; it does not prove which reading is wrong.

In [ ]:
ordered_readings.createOrReplaceTempView("ordered_readings")
sequenced = spark.sql("""
    SELECT vehicle_id, event_time, event_date, odometer_km,
           LAG(odometer_km) OVER (
               PARTITION BY vehicle_id, event_date ORDER BY event_time
           ) AS previous_odometer
    FROM ordered_readings
""")
sequenced = sequenced.withColumn("delta_km", F.col("odometer_km") - F.col("previous_odometer"))
sequenced.filter(F.col("delta_km") < 0).show(truncate=False)

## List days we cannot trust
A day is untrusted if it has a decrease, a conflict, or a rejected record we can assign to that day. `collect_set` keeps distinct reasons when several problems affect one day. `sort_array` only makes their list order repeatable. Expected: three days.

In [ ]:
key_columns = ["vehicle_id", "event_date"]
# A decrease identifies an inconsistent DAY, not which individual reading is wrong.
decrease_days = (sequenced.filter(F.col("delta_km") < 0).select(*key_columns)
    .withColumn("issue", F.lit("odometer_decreased")))
conflict_days = conflicts.select(*key_columns).withColumn("issue", F.lit("conflicting_timestamp"))
rejected_days = (base_rejects
    .filter(F.col("vehicle_id").isNotNull() & (F.col("vehicle_id") != "") &
            F.col("event_date").between(start_date, end_date))
    .select(*key_columns).withColumn("issue", F.lit("rejected_reading_in_day")))
day_issues = (decrease_days.unionByName(conflict_days).unionByName(rejected_days)
    .groupBy(*key_columns).agg(F.sort_array(F.collect_set("issue")).alias("reasons")))
issue_days = day_issues.select(*key_columns).withColumn("has_day_issue", F.lit(True))

day_issues.show(truncate=False)

## Keep accepted Silver readings
Exclude every reading on a problem day using the day keys. This avoids keeping a spike just because the next reading went down.

In [ ]:
accepted = sequenced.join(issue_days.select(*key_columns), key_columns, "left_anti")
print("Accepted Silver readings:", accepted.count())

## Keep rejected and held records
This inspection table uses a reasons list for both types of record. A rejected record gets its first failure; a held record gets the day’s reasons. `unionByName` lines up the columns; missing audit fields on held rows become null. Expected: four rejected + eighteen held.

In [ ]:
# Hold all usable-looking readings from an untrusted day; do not accuse one row.
held_readings = unique_readings.join(day_issues, key_columns, "inner")
quarantine_columns = business_columns + ["reasons"]
row_quarantine = (base_rejects.withColumn("reasons", F.array("reject_reason"))
    .select(*quarantine_columns, "event_timestamp", "_corrupt_record", "source_file")
    .withColumn("quarantine_scope", F.lit("record")))
day_quarantine = held_readings.select(*quarantine_columns).withColumn("quarantine_scope", F.lit("day_hold"))
quarantine = row_quarantine.unionByName(day_quarantine, allowMissingColumns=True)
quarantine.groupBy("quarantine_scope").count().show()

## Calculate daily distance
One row per vehicle/day. Calculate max minus min only when the day passed the checks and has at least two readings. Two equal readings give zero; missing or unreliable readings give null.

In [ ]:
daily_observations = (sequenced.groupBy(*key_columns).agg(
    F.count("*").alias("reading_count"),
    F.min("event_time").alias("first_reading"),
    F.max("event_time").alias("last_reading"),
    (F.max("odometer_km") - F.min("odometer_km")).alias("observed_change_km")))
# A full join retains days whose only readings were rejected.
daily = (daily_observations.join(issue_days, key_columns, "full")
    .withColumn("reading_count", F.coalesce("reading_count", F.lit(0))))
daily = daily.withColumn("distance_status",
    F.when(F.col("has_day_issue"), "untrusted")
     .when(F.col("reading_count") < 2, "insufficient_readings")
     .otherwise("trusted"))
daily = (daily.withColumn("observed_distance_km",
    F.when(F.col("distance_status") == "trusted", F.round("observed_change_km", 2)))
    .drop("observed_change_km", "has_day_issue")
    .filter(F.col("event_date").between(start_date, end_date)))

daily.groupBy("distance_status").count().show()

## Save the telemetry results
Save the day issues, accepted readings, quarantine and daily observations as separate Delta tables.

In [ ]:
day_issues.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(output_dir / "delta" / "silver_day_issues"))
accepted.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(output_dir / "delta" / "silver_telemetry"))
quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(output_dir / "delta" / "quarantine_telemetry"))
daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(output_dir / "delta" / "daily_observations"))
daily = spark.read.format("delta").load(str(output_dir / "delta" / "daily_observations"))
daily.orderBy("event_date", "vehicle_id").show(8, truncate=False)

## Check and save the fleet roster
Reject an empty roster, duplicate vehicle IDs or incomplete records. Save the checked roster for Gold.

In [ ]:
if fleet.count() == 0:
    raise ValueError("Fleet database is empty")
if fleet.select("vehicle_id").distinct().count() != fleet.count():
    raise ValueError("Duplicate fleet vehicle ID")
incomplete_fleet = fleet.filter(
    F.col("vehicle_id").isNull() | (F.col("vehicle_id") == "") |
    F.col("model").isNull() | F.col("depot_id").isNull() |
    F.col("depot_name").isNull() | F.col("latitude").isNull() |
    F.col("longitude").isNull())
if incomplete_fleet.count() != 0:
    raise ValueError("Incomplete fleet record")
fleet.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(output_dir / "delta" / "silver_fleet"))
print("Fleet rows checked and saved:", fleet.count())

## Check and save weather
Expected: one row per depot per day. Stop if coverage is incomplete or a depot/date key is duplicated.

In [ ]:
expected_weather_rows = fleet.select("depot_id").distinct().count() * ((end_date - start_date).days + 1)
if weather.count() != expected_weather_rows:
    raise ValueError(f"Expected {expected_weather_rows} depot-days of weather")
if weather.select("depot_id", "event_date").distinct().count() != weather.count():
    raise ValueError("Duplicate weather depot/date key")
weather.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(output_dir / "delta" / "silver_weather"))
weather = spark.read.format("delta").load(str(output_dir / "delta" / "silver_weather"))
weather.show(4, truncate=False)

## Next: 03_gold_activity
Silver tables are saved. Gold will load them in its own session. A failed Silver run must be fixed before running Gold; table writes are not one transaction.